# 04. Modelado y Optimización de Hiperparámetros (KNN y Random Forest)

**Universidad Nacional Federico Villarreal (UNFV)**  
**Facultad de Ingeniería Electrónica e Informática (FIEI)**  
**Autor:** Jimenez Villalva Franz José  

Entrenamiento de la línea base (Dummy), optimización del hiperparámetro k en KNN y ensamble Random Forest.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import joblib

DATA_PATH = '../data/processed/anemia_infantil_peru_endes.csv'
df = pd.read_csv(DATA_PATH)
X = df[['Hemoglobina_Observada', 'Altitud_msnm', 'Edad_Meses', 'Sexo', 'Area_Residencia', 'Peso_kg', 'Talla_cm']]
y = df['Anemia']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### 1. Modelo 1: Baseline Dummy (Línea Base Trivial)


In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_scaled, y_train)
print('Baseline Dummy entrenado.')


### 2. Modelo 2: Optimización y Entrenamiento de KNN


In [ ]:
k_range = list(range(1, 32, 2))
f1_scores = []
for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_temp.fit(X_train_scaled, y_train)
    f1_scores.append(f1_score(y_test, knn_temp.predict(X_test_scaled)))

best_k = k_range[np.argmax(f1_scores)]
print(f'Mejor k: {best_k} (F1 = {max(f1_scores):.4f})')

knn_best = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn_best.fit(X_train_scaled, y_train)
joblib.dump(knn_best, '../models/modelo_knn.joblib')


### 3. Modelo 3: Ensamble Random Forest (100 Árboles)


In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
rf.fit(X_train_scaled, y_train)
joblib.dump(rf, '../models/modelo_rf.joblib')
print('Random Forest entrenado y serializado en models/modelo_rf.joblib')
